In [1]:
pip install sqlalchemy>=2.0.30

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Step 2: Bootstrapping the Corporate Database
import sqlite3
import pandas as pd

print("="*60)
print(" INITIATING TEXT-TO-SQL AGENT ORCHESTRATION ")
print("="*60)

print("[System] Bootstrapping local SQLite Data Warehouse...")

# 1. Connect to a local SQLite file (creates it if it doesn't exist)
conn = sqlite3.connect("corporate_warehouse.db")
cursor = conn.cursor()

# 2. The DDL: Creating the executive_financials table
cursor.execute('''
CREATE TABLE IF NOT EXISTS executive_financials (
    transaction_id TEXT PRIMARY KEY,
    department_name TEXT,
    fiscal_quarter TEXT,
    revenue_usd REAL,
    operational_costs REAL,
    approval_status TEXT
)
''')

# 3. Injecting the Mock Data
mock_data = [
    ('TXN-001', 'Engineering', 'Q3', 1500000.00, 800000.00, 'APPROVED'),
    ('TXN-002', 'Marketing', 'Q3', 450000.00, 600000.00, 'APPROVED'), # Note the loss here
    ('TXN-003', 'Sales', 'Q3', 3200000.00, 400000.00, 'APPROVED'),
    ('TXN-004', 'Engineering', 'Q4', 2100000.00, 950000.00, 'PENDING'),
    ('TXN-005', 'HR', 'Q4', 0.00, 120000.00, 'APPROVED')
]

cursor.executemany('''
INSERT OR IGNORE INTO executive_financials
VALUES (?, ?, ?, ?, ?, ?)
''', mock_data)

conn.commit()
conn.close()

print("[System] Database 'corporate_warehouse.db' is online and populated.")

 INITIATING TEXT-TO-SQL AGENT ORCHESTRATION 
[System] Bootstrapping local SQLite Data Warehouse...
[System] Database 'corporate_warehouse.db' is online and populated.


In [4]:
# Step 3: Wiring the SQL Agent
import os
from langchain_community.utilities import SQLDatabase
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.agent_toolkits import create_sql_agent

# Secure Authentication
from dotenv import load_dotenv
import os
load_dotenv()
api_key=os.getenv("GEMINI_API_KEY")


# 1. Connect LangChain to the SQLite database
db = SQLDatabase.from_uri("sqlite:///corporate_warehouse.db")

# 2. Initialize the AI Compiler (Gemini 2.5 Flash)
# We set temperature to 0.0 because SQL requires strict determinism, not creativity.
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.0)

print("[System] Connecting AI Agent to the Database Schema...")

# 3. Create the Autonomous SQL Agent
# 'verbose=True' is CRITICAL for teaching. It lets students see the Agent's thought process,
# the exact SQL it writes, and how it handles errors in real-time.
sql_agent = create_sql_agent(
    llm=llm,
    db=db,
    agent_type="zero-shot-react-description",
    verbose=True,
    max_iterations = 30 # Our Circuit Breaker from Part 4!
)

print("[System] Agent online. Ready for natural language queries.\n")

[System] Connecting AI Agent to the Database Schema...
[System] Agent online. Ready for natural language queries.



In [5]:
executive_query = (
    "Calculate the total net profit (revenue minus costs) across all departements for Q3. Only include approved transactions."
)

print("="*40)
print(f"User Query:'{executive_query}'")
print("="*40)

#Invoke the agent. Watch the verbose input in the terminal as it writes the SQl!
final_answer = sql_agent.invoke({"input": executive_query})
print("\n" + "="*40)

print("FINAL SYNTHESIZED ANSWER")
print("="*40)
print(final_answer['output'])


User Query:'Calculate the total net profit (revenue minus costs) across all departements for Q3. Only include approved transactions.'


> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Action Input: executive_financialsAction: sql_db_schema
Action Input: executive_financials
CREATE TABLE executive_financials (
	transaction_id TEXT, 
	department_name TEXT, 
	fiscal_quarter TEXT, 
	revenue_usd REAL, 
	operational_costs REAL, 
	approval_status TEXT, 
	PRIMARY KEY (transaction_id)
)

/*
3 rows from executive_financials table:
transaction_id	department_name	fiscal_quarter	revenue_usd	operational_costs	approval_status
TXN-001	Engineering	Q3	1500000.0	800000.0	APPROVED
TXN-002	Marketing	Q3	450000.0	600000.0	APPROVED
TXN-003	Sales	Q3	3200000.0	400000.0	APPROVED
*/Action: sql_db_query_checker
Action Input: SELECT SUM(revenue_usd - operational_costs) AS total_net_profit FROM executive_financials WHERE fiscal_quarter = 'Q3' AND approval_status = 'APPROVED';```sql
SELECT SUM(r